# Assignment 2: Scraping Dean Jarley's Blog

Following `tutorials/ucf_news_scraping.ipynb`, this notebook uses `curl_cffi`
to request HTML, BeautifulSoup to find the information, and pandas to save it.
It collects posts linked on the first landing page only, then visits each post
for its text and comments.


In [13]:
from pathlib import Path
from time import sleep
from urllib.parse import urljoin, urldefrag

from bs4 import BeautifulSoup
from curl_cffi import requests
import pandas as pd

BLOG_URL = "https://pauljarley.wordpress.com/"
COLUMNS = ["title", "url", "entry_date", "post_text", "comments"]


def get_soup(url):
    """Request a page and turn its HTML into a searchable BeautifulSoup object."""
    response = requests.get(url, impersonate="chrome124", timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html5lib")


## 1. Collect the title, URL, and date from the first page

Each WordPress post is an `article`. Its `entry-title` contains the post link,
and the `datetime` attribute of its `time` element supplies the full date,
including the time-zone offset.


In [14]:
landing_soup = get_soup(BLOG_URL)
data = []
seen_urls = set()

for article in landing_soup.select("article"):
    title_link = article.select_one(".entry-title a[href]")
    if title_link is None:
        continue

    post_url = urljoin(BLOG_URL, title_link["href"])
    if post_url in seen_urls:
        continue

    date_tag = article.select_one("time.entry-date[datetime], time[datetime]")
    if date_tag is None:
        raise ValueError(f"No publication date found for {post_url}")

    seen_urls.add(post_url)
    data.append({
        "title": title_link.get_text(" ", strip=True),
        "url": post_url,
        "entry_date": date_tag["datetime"],
        "post_text": "",
        "comments": "",
    })

if not data:
    raise ValueError("No posts found. Check the landing page's HTML selectors.")

article_df = pd.DataFrame(data, columns=COLUMNS)
print(f"Found {len(article_df)} posts on the first page.")
article_df[["title", "url", "entry_date"]].head()


Found 10 posts on the first page.


,title,url,entry_date
0,We Should All Toss Something into the Tip Jar,https://pauljarley.wordpress.com/2024/10/21/we...,2024-10-21T03:04:00-04:00
1,Our Failure Competition Starts Today,https://pauljarley.wordpress.com/2024/10/14/ou...,2024-10-14T14:05:46-04:00
2,Why I Didn’t Hire You,https://pauljarley.wordpress.com/2024/10/07/wh...,2024-10-07T13:51:09-04:00
3,Your Friends Are Redundant,https://pauljarley.wordpress.com/2024/09/30/yo...,2024-09-30T01:45:00-04:00
4,The Invitational is Friday,https://pauljarley.wordpress.com/2024/09/23/th...,2024-09-23T05:42:00-04:00


## 2. Visit each post and collect its text and comments

Only the post's `entry-content` is used for the post text. Sharing buttons and
related-post widgets are removed. Each comment is collected from its own
`comment-body`, so nested replies are not counted twice. Author and text are
combined as `Author: Comment`, with a newline between comments. Posts without
comments receive an empty string. Older comment pages are followed when present.


In [15]:
def get_comments(post_soup, post_url):
    """Collect comments and replies, including older comment pages if present."""
    comments = []
    seen_comment_ids = set()
    seen_pages = {urldefrag(post_url)[0]}
    page_soup = post_soup
    page_url = post_url

    while True:
        for body in page_soup.select(".commentlist article.comment, .comment-list article.comment"):
            # Use the nearest comment container's ID to avoid duplicates across pages.
            container = body.find_parent("li", id=True)
            comment_id = body.get("id") or (container.get("id") if container else None)
            if comment_id and comment_id in seen_comment_ids:
                continue

            author = body.select_one(".comment-author .fn, .comment-author .says + a")
            if author is None:
                author = body.select_one(".comment-author")
            content = body.select_one(".comment-content")
            if content is None:
                continue

            author_text = author.get_text(" ", strip=True) if author else "Anonymous"
            comment_text = content.get_text(" ", strip=True)
            comments.append(f"{author_text}: {comment_text}")
            if comment_id:
                seen_comment_ids.add(comment_id)

        older_link = page_soup.select_one(
            "#comment-nav-below .nav-previous a[href], "
            "#comment-nav-above .nav-previous a[href], "
            ".comment-navigation .nav-previous a[href]"
        )
        if older_link is None:
            break
        older_url = urldefrag(urljoin(page_url, older_link["href"]))[0]
        if older_url in seen_pages:
            break
        seen_pages.add(older_url)
        sleep(1)
        page_soup = get_soup(older_url)
        page_url = older_url

    return "\n".join(comments)


def get_post_info(post_soup, post_url):
    article = post_soup.select_one("article")
    content = article.select_one(".entry-content") if article else None
    if content is None:
        raise ValueError(f"No post text found for {post_url}")

    for widget in content.select(
        ".sharedaddy, .jp-relatedposts, #jp-relatedposts, script, style"
    ):
        widget.decompose()

    post_text = content.get_text(" ", strip=True)
    return post_text, get_comments(post_soup, post_url)


for index, row in article_df.iterrows():
    print(f"Collecting {index + 1}/{len(article_df)}: {row['title']}")
    sleep(1)
    post_soup = get_soup(row["url"])
    post_text, comments = get_post_info(post_soup, row["url"])
    article_df.loc[index, "post_text"] = post_text
    article_df.loc[index, "comments"] = comments

article_df.head()


,title,url,entry_date,post_text,comments
0,We Should All Toss Something into the Tip Jar,https://pauljarley.wordpress.com/2024/10/21/we...,2024-10-21T03:04:00-04:00,Last Thursday I had the honor of attending the...,"Jeff Bowers: And that sir, is just one of the ..."
1,Our Failure Competition Starts Today,https://pauljarley.wordpress.com/2024/10/14/ou...,2024-10-14T14:05:46-04:00,"For several years, we have asked students to t...",
2,Why I Didn’t Hire You,https://pauljarley.wordpress.com/2024/10/07/wh...,2024-10-07T13:51:09-04:00,"If you are graduating in December, there is a ...",
3,Your Friends Are Redundant,https://pauljarley.wordpress.com/2024/09/30/yo...,2024-09-30T01:45:00-04:00,You have all heard the expression: “It is not ...,
4,The Invitational is Friday,https://pauljarley.wordpress.com/2024/09/23/th...,2024-09-23T05:42:00-04:00,This is the one time in your life when employe...,


## 3. Save as csv

UTF-8 with a BOM helps Excel
display punctuation correctly; pandas quotes fields containing commas or newlines.
The path below supports starting Jupyter from this assignment folder or any of
its parent folders inside the repository. A failed request or missing required
post field stops execution so that problems are visible before saving the CSV.


In [16]:
def find_assignment_folder():
    cwd = Path.cwd().resolve()
    for folder in (cwd, *cwd.parents):
        if (folder / "dean_scraping.ipynb").is_file():
            return folder
        candidate = folder / "assignments" / "submissions" / "assignment_2"
        if (candidate / "dean_scraping.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter from within the repository folder.")


if article_df["post_text"].str.strip().eq("").any():
    raise ValueError("At least one post has no text. Review the scraping results.")

output_file = find_assignment_folder() / "dean_scraping.csv"
article_df.to_csv(output_file, columns=COLUMNS, encoding="utf-8-sig", index=False)
print(f"Saved {len(article_df)} posts to {output_file}")


Saved 10 posts to C:\Research\Subjects\content analysis\MAN7916-main-testing\assignments\submissions\assignment_2\dean_scraping.csv
